# NER Data Augmentation - Baseline Experiment
## Full vs. 10% Subsampled BERT Training on CoNLL-2003

This notebook runs a baseline comparison experiment:
- **Model 1**: BERT fine-tuned on the full CoNLL-2003 training dataset (~14k samples)
- **Model 2**: BERT fine-tuned on 10% subsampled training dataset (~1.4k samples)

Both models are evaluated on the same test set to measure the trade-off between training efficiency and model performance.

**Designed for Google Colab** - Takes advantage of Colab's GPU acceleration (recommended: T4 or A100)
**No Google Drive mounting required** - Uses Colab's session storage directly

In [1]:
import os
# Install required packages
print("Installing required packages...")

packages = [
    # "torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118",
    "torch>=2.0.0",
    "torchvision>=0.15.0",
    "torchaudio>=2.0.0",
    "datasets==2.14.7",
    "huggingface-hub==0.36.2",
    "transformers==4.57.6",
    "pandas>=1.3.0",
    "numpy>=1.21.0,<1.24.0",
    "pytest>=7.0.0",
    "accelerate>=0.20.0",
    "scikit-learn>=1.0.0",
    "seqeval>=1.2.2",
    "nltk>=3.8"
]

for package in packages:
    pkg_name = package.split()[0]
    print(f"Installing {pkg_name}...")
    # Force reinstall huggingface-hub and datasets to avoid version conflicts
    if "huggingface-hub" in package or "datasets" in package:
        os.system(f"pip install -q {package} --force-reinstall")
    else:
        os.system(f"pip install -q {package}")

print("All packages installed successfully!")

Installing required packages...
Installing torch>=2.0.0...
Installing torchvision>=0.15.0...
Installing torchaudio>=2.0.0...
Installing datasets==2.14.7...
Installing huggingface-hub==0.36.2...
Installing transformers==4.57.6...
Installing pandas>=1.3.0...
Installing numpy>=1.21.0,<1.24.0...
Installing pytest>=7.0.0...
Installing accelerate>=0.20.0...
Installing scikit-learn>=1.0.0...
Installing seqeval>=1.2.2...
Installing nltk>=3.8...
All packages installed successfully!


Seeding for reproducibility:

In [2]:
import random
import torch
import numpy as np

# Set random seeds for reproducibility
RANDOM_SEED = 17

random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)

# For CUDA reproducibility (if GPU is available)
if torch.cuda.is_available():
    torch.cuda.manual_seed(RANDOM_SEED)
    torch.cuda.manual_seed_all(RANDOM_SEED)
    # Set CUDA to be deterministic
    torch.backends.cuda.deterministic = True
    torch.backends.cuda.benchmark = False

print(f"Random seeds set to {RANDOM_SEED} for reproducibility")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA device: {torch.cuda.get_device_name(0)}")

Random seeds set to 17 for reproducibility
CUDA available: True
CUDA device: Tesla T4


In [3]:
import nltk

# Download necessary NLTK resources for augmentation
print("Downloading NLTK resources...")
nltk.download('wordnet', quiet=True)
nltk.download('averaged_perceptron_tagger', quiet=True)
nltk.download('punkt', quiet=True)
print("NLTK resources downloaded successfully!")

NLTK resources downloaded successfully!


The two cells below clone the code from GitHub and set working directories

In [ ]:
GITHUB_REPO = "https://github.com/jtco19/ner_data_augmentation_error_analysis.git"
BRANCH = "main"  # Change this to clone from a different branch

print("=" * 70)
print(f"STEP 1: Cloning repository from GitHub (branch: {BRANCH})...")
print("=" * 70)
!git clone --branch {BRANCH} {GITHUB_REPO}
%cd ner_data_augmentation_error_analysis/src

print(f"Working directory: {os.getcwd()}")

STEP 1: Cloning repository from GitHub (branch: boyan-metric-review)...
fatal: destination path 'ner_data_augmentation_error_analysis' already exists and is not an empty directory.
/content/ner_data_augmentation_error_analysis/src
Working directory: /content/ner_data_augmentation_error_analysis/src


In [5]:
import sys
import logging
import json
import time
from typing import Dict, Tuple, List
from pathlib import Path

# Configure logging
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - %(name)s - %(levelname)s - %(message)s"
)
logger = logging.getLogger(__name__)

# Find and configure path for source directories
# Works for both local execution and Colab
def find_source_dirs():
    """Find data and model directories, handling both local and Colab environments."""
    # List of possible locations to check
    search_paths = [
        Path("."),  # Current directory (local default)
        Path("./src"),  # src subfolder
        Path("/content"),  # Colab root
        Path("/content/src"),  # Colab with src folder structure
    ]
    
    data_dir = None
    model_dir = None
    metrics_dir = None
    base_dir = None
    
    # Search for the source directories
    for search_path in search_paths:
        if search_path.exists():
            data_candidate = search_path / "data"
            model_candidate = search_path / "model"
            metrics_candidate = search_path / "metrics"
            
            if data_candidate.exists() and model_candidate.exists() and metrics_candidate.exists():
                data_dir = data_candidate
                model_dir = model_candidate
                metrics_dir = metrics_candidate
                base_dir = search_path
                break
    
    if data_dir is None or model_dir is None or metrics_dir is None:
        raise FileNotFoundError(
            "Could not find 'data', 'model', and 'metrics' directories. "
            "Please ensure project files are uploaded correctly. "
            f"Checked: {[str(p) for p in search_paths]}"
        )
    
    return data_dir, model_dir, metrics_dir, base_dir

# Find directories
data_dir, model_dir, metrics_dir, base_dir = find_source_dirs()

# Add to path
sys.path.insert(0, str(data_dir))
sys.path.insert(0, str(model_dir))
sys.path.insert(0, str(metrics_dir))

print(f"Base directory: {base_dir}")
print(f"Data directory: {data_dir}")
print(f"Model directory: {model_dir}")
print(f"Metrics directory: {metrics_dir}")
print(f"Python path configured")

Base directory: .
Data directory: data
Model directory: model
Metrics directory: metrics
Python path configured


If the following cell throws an `ImportError`, restart the kernel and rerun the entire notebook (currently trying to identify cause of error and why it occurs randomly)

In [6]:
from load_conll2003 import load_conll2003_dataset, subsample_dataset
from bert_model import create_bert_ner_model
from augmentation import augment_naive_eda, augment_entity_aware, augment_contextual_mlm, augment_back_translation
from metrics import per_type_f1_scores

print("Modules imported successfully!")

Modules imported successfully!


In [7]:
# Load CoNLL-2003 dataset directly using safe loader
print("Loading CoNLL-2003 dataset...")

# Use the inline loader that bypasses scripts
dataset = load_conll2003_dataset()

print(f"\nDataset loaded successfully!")
print(f"Dataset splits: {dataset.keys()}")
print(f"  Train: {len(dataset['train'])} samples")
print(f"  Validation: {len(dataset['validation'])} samples")
print(f"  Test: {len(dataset['test'])} samples")

# Create subsampled dataset (10% of training)
print("\nCreating 10% subsampled dataset...")

subsampled_dataset = subsample_dataset(dataset, sample_rate=0.1)

print(f"Subsampled dataset splits:")
print(f"  Train: {len(subsampled_dataset['train'])} samples")
print(f"  Validation: {len(subsampled_dataset['validation'])} samples")
print(f"  Test: {len(subsampled_dataset['test'])} samples")

Loading CoNLL-2003 dataset...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(



Dataset loaded successfully!
Dataset splits: dict_keys(['train', 'validation', 'test'])
  Train: 14041 samples
  Validation: 3250 samples
  Test: 3453 samples

Creating 10% subsampled dataset...
Subsampled train: 14041 -> 1404 samples (10.0%)
Subsampled validation: 3250 -> 325 samples (10.0%)
Subsampled test: 3453 -> 345 samples (10.0%)
Subsampled dataset splits:
  Train: 1404 samples
  Validation: 325 samples
  Test: 345 samples


## Data Augmentation & Quality Analysis

Create augmented versions of train/validation datasets and measure augmentation quality metrics (span corruption rate, label flip rate)


Helper Functions:

In [8]:
# Build label mappings from dataset features
def build_label_mappings():
    """Build int-to-str and str-to-int mappings for all tag types."""
    # For Sequence features, we need to access .feature.names
    ner_feature = dataset['train'].features['ner_tags']
    pos_feature = dataset['train'].features['pos_tags']
    chunk_feature = dataset['train'].features['chunk_tags']
    
    # Extract names from the feature (handling Sequence wrapper)
    if hasattr(ner_feature, 'feature'):  # It's a Sequence
        ner_names = ner_feature.feature.names
        pos_names = pos_feature.feature.names
        chunk_names = chunk_feature.feature.names
    else:  # Direct ClassLabel
        ner_names = ner_feature.names
        pos_names = pos_feature.names
        chunk_names = chunk_feature.names
    
    return {
        'ner': {
            'int2str': {i: name for i, name in enumerate(ner_names)},
            'str2int': {name: i for i, name in enumerate(ner_names)}
        },
        'pos': {
            'int2str': {i: name for i, name in enumerate(pos_names)},
            'str2int': {name: i for i, name in enumerate(pos_names)}
        },
        'chunk': {
            'int2str': {i: name for i, name in enumerate(chunk_names)},
            'str2int': {name: i for i, name in enumerate(chunk_names)}
        }
    }

label_mappings = build_label_mappings()

# Build synonym dictionary using NLTK WordNet
def build_synonym_dict(sample_size=100):
    """Build a synonym dictionary from WordNet for common words."""
    synonym_dict = {}
    
    # Sample words from training data
    word_samples = set()
    for i, sample in enumerate(baseline_dataset['train']):
        if i >= sample_size:
            break
        for token in sample['tokens']:
            word = token.lower()
            if len(word) > 2 and word.isalpha():  # Filter short/non-alpha words
                word_samples.add(word)
    
    # Build synonyms for sampled words
    for word in word_samples:
        synonyms = set()
        for synset in wordnet.synsets(word):
            for lemma in synset.lemmas():
                if lemma.name() != word and '_' not in lemma.name():
                    synonyms.add(lemma.name())
        if synonyms:
            synonym_dict[word] = list(synonyms)[:5]  # Limit to top 5 synonyms
    
    print(f"Built synonym dictionary with {len(synonym_dict)} words")
    return synonym_dict

# Build entity knowledge base from training data
def build_entity_kb():
    """Build entity knowledge base by extracting entities from training data."""
    from metrics import extract_entities_from_labels
    
    entity_kb = defaultdict(list)
    
    for sample in baseline_dataset['train']:
        tokens = sample['tokens']
        ner_labels = [label_mappings['ner']['int2str'].get(tag, 'O') for tag in sample['ner_tags']]
        
        # Extract entities from this sample
        entities = extract_entities_from_labels(tokens, ner_labels)
        
        for start_idx, end_idx, entity_type in entities:
            entity_span = tokens[start_idx:end_idx+1]
            # Store the tokenized span for replacement
            entity_kb[entity_type].append(entity_span)
    
    print(f"Built entity KB with {len(entity_kb)} entity types")
    for ent_type, spans in entity_kb.items():
        print(f"  {ent_type}: {len(spans)} examples")
    
    return dict(entity_kb)

# Wrapper function to handle HuggingFace fill-mask pipeline output
def create_mlm_wrapper(hf_pipeline):
    """
    Wrapper that converts HuggingFace fill-mask pipeline output to a simple string token.
    The HF pipeline returns a list of dicts with predictions, we extract the top prediction.
    """
    def wrapped_pipeline(masked_sentence):
        try:
            predictions = hf_pipeline(masked_sentence)
            
            # Handle list of predictions (HF format)
            if isinstance(predictions, list) and len(predictions) > 0:
                top_prediction = predictions[0]
                if isinstance(top_prediction, dict):
                    # Extract token string and clean up
                    predicted_word = top_prediction.get('token_str', '').strip()
                    # Remove BERT's ## prefix for subword pieces
                    if predicted_word.startswith('##'):
                        predicted_word = predicted_word[2:]
                    return predicted_word
                else:
                    return str(top_prediction)
            else:
                return str(predictions)
        except Exception as e:
            logger.warning(f"MLM pipeline error: {e}. Returning '[UNK]'")
            return '[UNK]'
    
    return wrapped_pipeline

# Initialize MLM pipeline (optional - for contextual_mlm augmentation)
def build_mlm_pipeline():
    """Build MLM pipeline for contextual replacement. Optional - returns None if not available."""
    try:
        from transformers import pipeline
        print("Initializing MLM pipeline...")
        hf_pipeline = pipeline("fill-mask", model="bert-base-uncased")
        mlm_pipeline = create_mlm_wrapper(hf_pipeline)
        print("✓ MLM pipeline initialized successfully")
        return mlm_pipeline
    except Exception as e:
        print(f"⚠ MLM pipeline initialization failed: {e}")
        print("  Contextual MLM augmentation will be skipped")
        return None

# Initialize back-translation augmentation helpers (optional)
def build_back_translation_helpers():
    """Build NMT translation functions and alignment heuristic. Optional - returns None if not available."""
    try:
        from transformers import pipeline
        print("Initializing back-translation NMT pipelines...")
        
        device = "cuda" if torch.cuda.is_available() else "cpu"
        print(f"  Using device: {device}")
        
        # Initialize translation pipelines
        translator_en_to_de = pipeline(
            "translation_en_to_de", 
            model="Helsinki-NLP/opus-mt-en-de",
            device=0 if device == "cuda" else -1
        )
        translator_de_to_en = pipeline(
            "translation_de_to_en", 
            model="Helsinki-NLP/opus-mt-de-en",
            device=0 if device == "cuda" else -1
        )
        
        # NMT forward function: English -> German
        def nmt_forward(text):
            try:
                result = translator_en_to_de(text, max_length=512)
                if isinstance(result, list) and len(result) > 0:
                    return result[0].get('translation_text', text)
                return text
            except Exception as e:
                logger.warning(f"Forward translation failed: {e}")
                return text
        
        # NMT backward function: German -> English
        def nmt_backward(text):
            try:
                result = translator_de_to_en(text, max_length=512)
                if isinstance(result, list) and len(result) > 0:
                    return result[0].get('translation_text', text)
                return text
            except Exception as e:
                logger.warning(f"Backward translation failed: {e}")
                return text
        
        # Alignment heuristic: simple position-based alignment
        def alignment_heuristic(original_tokens, original_labels, augmented_tokens):
            """
            Simple alignment heuristic: preserve labels based on position and length.
            """
            aug_labels = []
            if len(augmented_tokens) == len(original_tokens):
                # Same length: direct copy
                aug_labels = original_labels
            elif len(augmented_tokens) < len(original_tokens):
                # Shorter: truncate labels
                aug_labels = original_labels[:len(augmented_tokens)]
            else:
                # Longer: repeat labels by extending entity spans
                label_idx = 0
                for aug_idx in range(len(augmented_tokens)):
                    ratio = aug_idx / len(augmented_tokens)
                    orig_idx = min(int(ratio * len(original_tokens)), len(original_tokens) - 1)
                    aug_labels.append(original_labels[orig_idx])
            
            return aug_labels
        
        print("✓ Back-translation NMT pipelines initialized successfully")
        return nmt_forward, nmt_backward, alignment_heuristic
        
    except Exception as e:
        print(f"⚠ Back-translation initialization failed: {e}")
        print("  Back-translation augmentation will be skipped")
        return None, None, None

Check for preexisting augmented datasets (so we can avoid generating the datasets again if present)

In [9]:
import os
from pathlib import Path
from copy import deepcopy
from datasets import DatasetDict, load_from_disk

# Check for pre-computed augmented datasets
augmented_datasets_dir = Path("./data/augmented_datasets")
augmented_datasets = {}

print("=" * 70)
print("CHECKING FOR PRE-COMPUTED AUGMENTED DATASETS")
print("=" * 70)

if augmented_datasets_dir.exists():
    print("\nChecking for pre-computed augmented datasets...")
    found_datasets = False
    for method_dir in augmented_datasets_dir.glob("*/"):
        method_name = method_dir.name
        try:
            baseline_dataset = deepcopy(dataset)
            augmented_datasets[method_name] = load_from_disk(str(method_dir))
            print(f"  ✓ Loaded {method_name} augmented dataset from {method_dir}")
            found_datasets = True
        except Exception as e:
            print(f"  ✗ Failed to load {method_name}: {e}")
    
    if found_datasets:
        print(f"\n✓ Successfully loaded {len(augmented_datasets)} pre-computed augmented datasets!")
        print("Augmentation step will be SKIPPED.")
    else:
        print("\nNo valid augmented datasets found in directory.")
        print("Augmentation will be created in the next cell.")
else:
    print(f"\nNo pre-computed augmented datasets directory found at {augmented_datasets_dir}")
    print("Augmentation will be created in the next cell.")

print("=" * 70 + "\n")

CHECKING FOR PRE-COMPUTED AUGMENTED DATASETS

Checking for pre-computed augmented datasets...
  ✓ Loaded entity_replacement augmented dataset from data/augmented_datasets/entity_replacement
  ✓ Loaded back_translation augmented dataset from data/augmented_datasets/back_translation
  ✓ Loaded contextual_mlm augmented dataset from data/augmented_datasets/contextual_mlm
  ✓ Loaded eda augmented dataset from data/augmented_datasets/eda

✓ Successfully loaded 4 pre-computed augmented datasets!
Augmentation step will be SKIPPED.



In [10]:
from nltk.corpus import wordnet
from collections import defaultdict

# Only run augmentation if datasets weren't already loaded
if not augmented_datasets:
    print("=" * 70)
    print("CREATING AUGMENTED DATASETS")
    print("=" * 70)

    # Store baseline (unaugmented) datasets
    baseline_dataset = deepcopy(dataset)

    print(f"\nBaseline dataset sizes:")
    print(f"  Train: {len(baseline_dataset['train'])} samples")
    print(f"  Validation: {len(baseline_dataset['validation'])} samples")
    print(f"  Test: {len(baseline_dataset['test'])} samples (held constant for all experiments)")

    # Build required data structures for augmentation
    print("\nBuilding augmentation resources...")
    synonym_dict = build_synonym_dict(sample_size=200)
    entity_kb = build_entity_kb()
    mlm_pipeline = build_mlm_pipeline()
    nmt_forward, nmt_backward, alignment_heuristic = build_back_translation_helpers()

    # Create augmented datasets using different methods
    augmentation_methods = ['eda', 'entity_replacement']

    # Add optional augmentation methods if their resources are available
    if nmt_forward is not None and nmt_backward is not None and alignment_heuristic is not None:
        augmentation_methods.append('back_translation')
    if mlm_pipeline is not None:
        augmentation_methods.append('contextual_mlm')

    print(f"\nAugmentation methods to be applied: {augmentation_methods}")

    augmented_datasets = {}

    for method in augmentation_methods:
        print(f"\nAugmenting dataset using: {method.upper()}")
        
        augmented_train = []
        augmented_val = []
        
        # Augment training set
        for idx, sample in enumerate(baseline_dataset['train']):
            if (idx + 1) % 2000 == 0:
                print(f"  Augmented {idx + 1}/{len(baseline_dataset['train'])} training samples")
            
            tokens = sample['tokens']
            ner_labels = [label_mappings['ner']['int2str'].get(tag, 'O') for tag in sample['ner_tags']]
            
            # Apply augmentation based on method
            if method == 'eda':
                aug_tokens, aug_labels = augment_naive_eda(
                    tokens=tokens,
                    labels=ner_labels,
                    synonym_dict=synonym_dict,
                    alpha=0.1
                )
            elif method == 'entity_replacement':
                aug_tokens, aug_labels = augment_entity_aware(
                    tokens=tokens,
                    labels=ner_labels,
                    entity_kb=entity_kb
                )
            elif method == 'contextual_mlm':
                aug_tokens, aug_labels = augment_contextual_mlm(
                    tokens=tokens,
                    labels=ner_labels,
                    mlm_pipeline=mlm_pipeline,
                    alpha=0.1
                )
            elif method == 'back_translation':
                # Apply back-translation using augment_back_translation function
                aug_tokens, aug_labels = augment_back_translation(
                    tokens=tokens,
                    labels=ner_labels,
                    nmt_forward=nmt_forward,
                    nmt_backward=nmt_backward,
                    alignment_heuristic=alignment_heuristic
                )
            
            # Convert back to HuggingFace format
            pos_tags = sample['pos_tags']
            chunk_tags = sample['chunk_tags']
            
            pos_labels = [label_mappings['pos']['int2str'].get(tag, 'NN') for tag in pos_tags]
            chunk_labels = [label_mappings['chunk']['int2str'].get(tag, 'O') for tag in chunk_tags]
            
            # Align pos and chunk tags to new token length
            pos_aligned = pos_labels[:len(aug_tokens)] if len(aug_tokens) <= len(pos_labels) else pos_labels + ['NN'] * (len(aug_tokens) - len(pos_labels))
            chunk_aligned = chunk_labels[:len(aug_tokens)] if len(aug_tokens) <= len(chunk_labels) else chunk_labels + ['O'] * (len(aug_tokens) - len(chunk_labels))
            
            # Convert to indices
            pos_indices = [label_mappings['pos']['str2int'].get(tag, 0) for tag in pos_aligned[:len(aug_tokens)]]
            chunk_indices = [label_mappings['chunk']['str2int'].get(tag, 0) for tag in chunk_aligned[:len(aug_tokens)]]
            ner_indices = [label_mappings['ner']['str2int'].get(tag, 0) for tag in aug_labels]
            
            augmented_sample = {
                'id': sample['id'],
                'tokens': aug_tokens,
                'pos_tags': pos_indices,
                'chunk_tags': chunk_indices,
                'ner_tags': ner_indices
            }
            augmented_train.append(augmented_sample)
        
        # Augment validation set
        for idx, sample in enumerate(baseline_dataset['validation']):
            tokens = sample['tokens']
            ner_labels = [label_mappings['ner']['int2str'].get(tag, 'O') for tag in sample['ner_tags']]
            
            if method == 'eda':
                aug_tokens, aug_labels = augment_naive_eda(
                    tokens=tokens,
                    labels=ner_labels,
                    synonym_dict=synonym_dict,
                    alpha=0.1
                )
            elif method == 'entity_replacement':
                aug_tokens, aug_labels = augment_entity_aware(
                    tokens=tokens,
                    labels=ner_labels,
                    entity_kb=entity_kb
                )
            elif method == 'contextual_mlm':
                aug_tokens, aug_labels = augment_contextual_mlm(
                    tokens=tokens,
                    labels=ner_labels,
                    mlm_pipeline=mlm_pipeline,
                    alpha=0.1
                )
            elif method == 'back_translation':
                # Apply back-translation using augment_back_translation function
                aug_tokens, aug_labels = augment_back_translation(
                    tokens=tokens,
                    labels=ner_labels,
                    nmt_forward=nmt_forward,
                    nmt_backward=nmt_backward,
                    alignment_heuristic=alignment_heuristic
                )
            
            pos_tags = sample['pos_tags']
            chunk_tags = sample['chunk_tags']
            
            pos_labels = [label_mappings['pos']['int2str'].get(tag, 'NN') for tag in pos_tags]
            chunk_labels = [label_mappings['chunk']['int2str'].get(tag, 'O') for tag in chunk_tags]
            
            pos_aligned = pos_labels[:len(aug_tokens)] if len(aug_tokens) <= len(pos_labels) else pos_labels + ['NN'] * (len(aug_tokens) - len(pos_labels))
            chunk_aligned = chunk_labels[:len(aug_tokens)] if len(aug_tokens) <= len(chunk_labels) else chunk_labels + ['O'] * (len(aug_tokens) - len(chunk_labels))
            
            pos_indices = [label_mappings['pos']['str2int'].get(tag, 0) for tag in pos_aligned[:len(aug_tokens)]]
            chunk_indices = [label_mappings['chunk']['str2int'].get(tag, 0) for tag in chunk_aligned[:len(aug_tokens)]]
            ner_indices = [label_mappings['ner']['str2int'].get(tag, 0) for tag in aug_labels]
            
            augmented_sample = {
                'id': sample['id'],
                'tokens': aug_tokens,
                'pos_tags': pos_indices,
                'chunk_tags': chunk_indices,
                'ner_tags': ner_indices
            }
            augmented_val.append(augmented_sample)
        
        # Create dataset dict with only augmented samples
        from datasets import Dataset, DatasetDict
        
        augmented_datasets[method] = DatasetDict({
            'train': Dataset.from_list(augmented_train),
            'validation': Dataset.from_list(augmented_val),
            'test': baseline_dataset['test']
        })
        
        print(f"  {method.upper()} augmented dataset sizes:")
        print(f"    Train: {len(augmented_train)} samples (augmented only)")
        print(f"    Validation: {len(augmented_val)} samples (augmented only)")
    
    print("\n" + "=" * 70)
    print("✓ Augmented datasets created successfully!")
    print("=" * 70)
else:
    print("\nAugmentation skipped - using pre-loaded augmented datasets from disk.")
    print("=" * 70)


Augmentation skipped - using pre-loaded augmented datasets from disk.


In [11]:
# Save augmented datasets to disk for future runs
import os
from pathlib import Path

augmented_datasets_dir = Path("./data/augmented_datasets")

# Create directory if it doesn't exist
augmented_datasets_dir.mkdir(parents=True, exist_ok=True)

print("\n" + "=" * 70)
print("SAVING AUGMENTED DATASETS TO DISK")
print("=" * 70)

if augmented_datasets:
    num_saved = 0
    num_already_exist = 0
    
    for method, dataset in augmented_datasets.items():
        save_path = augmented_datasets_dir / method
        
        # Check if dataset already exists
        if save_path.exists():
            print(f"⊘ Skipped '{method}' - already saved at {save_path}")
            num_already_exist += 1
        else:
            try:
                dataset.save_to_disk(str(save_path))
                print(f"✓ Saved '{method}' augmented dataset")
                print(f"  Location: {save_path}")
                num_saved += 1
            except Exception as e:
                print(f"✗ Failed to save '{method}' augmented dataset: {e}")
    
    print(f"\n✓ Summary: {num_saved} saved, {num_already_exist} already exist")
    if num_saved > 0:
        print(f"✓ New augmented datasets saved to {augmented_datasets_dir}")
    if num_already_exist > 0:
        print("⊘ Previously saved datasets were skipped")
    print("Datasets will be loaded from disk on the next run.")
else:
    print("No augmented datasets to save.")

print("=" * 70)


SAVING AUGMENTED DATASETS TO DISK
⊘ Skipped 'entity_replacement' - already saved at data/augmented_datasets/entity_replacement
⊘ Skipped 'back_translation' - already saved at data/augmented_datasets/back_translation
⊘ Skipped 'contextual_mlm' - already saved at data/augmented_datasets/contextual_mlm
⊘ Skipped 'eda' - already saved at data/augmented_datasets/eda

✓ Summary: 0 saved, 4 already exist
⊘ Previously saved datasets were skipped
Datasets will be loaded from disk on the next run.


## Augmentation Quality Metrics

Measure span corruption rate and label flip rate for augmented datasets


In [12]:
from metrics import span_corruption_rate, label_flip_rate, entity_token_perturbation_rate

# Measure quality of augmented datasets
corruption_metrics = {}

print("\n" + "="*70)
print("STEP 2: MEASURING AUGMENTATION QUALITY")
print("="*70)

for method, aug_dataset in augmented_datasets.items():
    print(f"\nAnalyzing {method.upper()} augmented dataset...")
    
    # Extract original (baseline) and augmented data
    # The augmented_datasets now contain only augmented samples (not combined)
    original_train_tokens = [sample['tokens'] for sample in baseline_dataset['train']]
    original_train_ner = [[label_mappings['ner']['int2str'].get(tag, 'O') for tag in sample['ner_tags']] 
                          for sample in baseline_dataset['train']]
    
    augmented_train_tokens = [sample['tokens'] for sample in aug_dataset['train']]
    augmented_train_ner = [[label_mappings['ner']['int2str'].get(tag, 'O') for tag in sample['ner_tags']]
                           for sample in aug_dataset['train']]
    
    # Compute span corruption rate
    print(f"  Computing span corruption rate...")
    span_corruption = span_corruption_rate(
        original_sentences=original_train_tokens,
        original_labels=original_train_ner,
        augmented_sentences=augmented_train_tokens,
        augmented_labels=augmented_train_ner
    )
    
    # Compute label flip rate
    print(f"  Computing label flip rate...")
    label_flip = label_flip_rate(
        original_labels=original_train_ner,
        augmented_labels=augmented_train_ner,
        original_sentences=original_train_tokens,
        augmented_sentences=augmented_train_tokens,
    )

    # Compute token perturbation rate for entity tokens
    print(f"  Computing token perturbation rate for entity tokens...")
    token_perturbation = entity_token_perturbation_rate(
        original_train_tokens,
        original_train_ner,
        augmented_train_tokens,
        augmented_train_ner,
    )
    
    corruption_metrics[method] = {
        'span_corruption': span_corruption,
        'label_flip': label_flip,
        'token_perturbation': token_perturbation
    }

    # Print results
    print(f"\n  {method.upper()} Augmentation Quality Metrics:")
    print(f"    Span Corruption Rate: {span_corruption['span_corruption_rate']:.4f} ({span_corruption['corrupted_sentences']}/{span_corruption['total_augmented_sentences']} sentences)")
    print(f"    Label Flip Rate: {label_flip['label_flip_rate']:.4f} ({label_flip['flipped_tokens']}/{label_flip['total_tokens']} tokens)")
    print(f"      - Entity → Non-Entity: {label_flip['entity_to_non_entity']}")
    print(f"      - Non-Entity → Entity: {label_flip['non_entity_to_entity']}")
    print(f"    Entity Token Perturbation Rate: {token_perturbation['entity_token_perturbation_rate']:.4f} ({token_perturbation['changed_or_deleted_entity_tokens']}/{token_perturbation['total_entity_tokens']} entity tokens)")

print("\n" + "="*70)


STEP 2: MEASURING AUGMENTATION QUALITY

Analyzing ENTITY_REPLACEMENT augmented dataset...
  Computing span corruption rate...
  Computing label flip rate...
  Computing token perturbation rate for entity tokens...

  ENTITY_REPLACEMENT Augmentation Quality Metrics:
    Span Corruption Rate: 0.4723 (11098/14041 sentences)
    Label Flip Rate: 0.0295 (6109/206758 tokens)
      - Entity → Non-Entity: 2972
      - Non-Entity → Entity: 3137
    Entity Token Perturbation Rate: 0.4676 (15920/34043 entity tokens)

Analyzing BACK_TRANSLATION augmented dataset...
  Computing span corruption rate...
  Computing label flip rate...
  Computing token perturbation rate for entity tokens...

  BACK_TRANSLATION Augmentation Quality Metrics:
    Span Corruption Rate: 0.4837 (5695/14041 sentences)
    Label Flip Rate: 0.0932 (19179/205750 tokens)
      - Entity → Non-Entity: 9771
      - Non-Entity → Entity: 9408
    Entity Token Perturbation Rate: 0.1931 (6573/34043 entity tokens)

Analyzing CONTEXTUAL

In [13]:
def train_model(
    model_name: str,
    dataset,
    num_epochs: int = 3,
    batch_size: int = 32,
    learning_rate: float = 2e-5,
    output_dir: str = None
) -> Tuple:
    """
    Train a BERT model on the provided dataset.
    
    Args:
        model_name: Name for logging purposes
        dataset: HuggingFace DatasetDict with 'train' and 'validation' splits
        num_epochs: Number of training epochs
        batch_size: Batch size for training
        learning_rate: Learning rate for optimizer
        output_dir: Directory to save the model
        
    Returns:
        Tuple of (trained_model, training_results_dict)
    """
    print(f"\n{'='*60}")
    print(f"Training: {model_name}")
    print(f"{'='*60}")
    
    # Create model
    model = create_bert_ner_model(num_labels=9)  # 9 NER tags
    
    # Log dataset sizes
    train_size = len(dataset["train"])
    val_size = len(dataset["validation"])
    print(f"Training samples: {train_size}")
    print(f"Validation samples: {val_size}")
    
    # Prepare dataset (tokenize and align labels)
    print("Preparing dataset...")
    train_dataset = model.prepare_dataset(dataset["train"])
    val_dataset = model.prepare_dataset(dataset["validation"])
    test_dataset = model.prepare_dataset(dataset["test"])
    
    # Train model
    print(f"Starting training for {num_epochs} epochs...")
    start_time = time.time()
    
    results = model.train(
        train_dataset=train_dataset,
        eval_dataset=val_dataset,
        output_dir=output_dir or f"./saved_models/{model_name}",
        num_epochs=num_epochs,
        per_device_batch_size=batch_size,
        learning_rate=learning_rate,
        weight_decay=0.01,
        warmup_steps=500,
        logging_steps=100,
        eval_strategy="epoch",
        save_strategy="epoch"
    )
    
    training_time = time.time() - start_time
    print(f"Training completed in {training_time:.2f} seconds")
    
    # Save model
    if output_dir:
        model.save_model(output_dir)
        print(f"Model saved to {output_dir}")
    
    return model, {
        "training_time": training_time,
        "train_samples": train_size,
        "val_samples": val_size
    }, test_dataset

print("Training function defined.")

Training function defined.


In [14]:
def evaluate_model(model, test_dataset, model_name: str) -> Dict:
    """
    Evaluate a trained model on the test set.
    
    Args:
        model: Trained BERT model
        test_dataset: Prepared (tokenized) test dataset
        model_name: Name for logging purposes
        
    Returns:
        Dictionary with evaluation metrics
    """
    print(f"\n{'='*60}")
    print(f"Evaluating: {model_name}")
    print(f"{'='*60}")
    
    test_size = len(test_dataset)
    print(f"Test samples: {test_size}")
    
    # Move model to CPU for evaluation (DirectML compatibility)
    model.model.to("cpu")
    
    # Evaluate
    print("Running evaluation...")
    start_time = time.time()
    
    eval_results = model.evaluate(test_dataset)
    
    eval_time = time.time() - start_time
    print(f"Evaluation completed in {eval_time:.2f} seconds")
    
    # Extract key metrics
    metrics = {
        "eval_time": eval_time,
        "test_samples": test_size,
        "eval_loss": eval_results.get("eval_loss", None),
        "eval_f1": eval_results.get("eval_f1", None),
        "eval_precision": eval_results.get("eval_precision", None),
        "eval_recall": eval_results.get("eval_recall", None),
    }
    
    return metrics

print("Evaluation function defined.")

Evaluation function defined.


In [15]:
def compare_performance(full_metrics: Dict, subsampled_metrics: Dict) -> None:
    """
    Compare performance metrics between full and subsampled models.
    
    Args:
        full_metrics: Evaluation metrics for full dataset model
        subsampled_metrics: Evaluation metrics for subsampled dataset model
    """
    print(f"\n{'='*60}")
    print("PERFORMANCE COMPARISON")
    print(f"{'='*60}\n")
    
    # Training metrics
    print("TRAINING METRICS:")
    print(f"  Full Dataset Model:")
    print(f"    - Training samples: {full_metrics.get('train_samples', 'N/A')}")
    print(f"    - Training time: {full_metrics.get('training_time', 'N/A'):.2f}s")
    
    print(f"  Subsampled (10%) Model:")
    print(f"    - Training samples: {subsampled_metrics.get('train_samples', 'N/A')}")
    print(f"    - Training time: {subsampled_metrics.get('training_time', 'N/A'):.2f}s")
    
    time_reduction = (
        (full_metrics.get('training_time', 0) - subsampled_metrics.get('training_time', 0)) /
        full_metrics.get('training_time', 1) * 100
    )
    print(f"  Time reduction: -{time_reduction:.1f}%\n")
    
    # Evaluation metrics
    print("EVALUATION METRICS:")
    print(f"  Full Dataset Model:")
    print(f"    - Eval Loss: {full_metrics.get('eval_loss', 'N/A'):.4f}")
    print(f"    - F1 Score: {full_metrics.get('eval_f1', 'N/A'):.4f}")
    print(f"    - Precision: {full_metrics.get('eval_precision', 'N/A'):.4f}")
    print(f"    - Recall: {full_metrics.get('eval_recall', 'N/A'):.4f}")
    
    print(f"  Subsampled (10%) Model:")
    print(f"    - Eval Loss: {subsampled_metrics.get('eval_loss', 'N/A'):.4f}")
    print(f"    - F1 Score: {subsampled_metrics.get('eval_f1', 'N/A'):.4f}")
    print(f"    - Precision: {subsampled_metrics.get('eval_precision', 'N/A'):.4f}")
    print(f"    - Recall: {subsampled_metrics.get('eval_recall', 'N/A'):.4f}")
    
    # Compute differences
    print("\nPERFORMANCE DELTA (Subsampled vs Full):")
    
    if full_metrics.get('eval_f1') and subsampled_metrics.get('eval_f1'):
        f1_delta = subsampled_metrics['eval_f1'] - full_metrics['eval_f1']
        print(f"  - F1 Score Delta: {f1_delta:+.4f} ({f1_delta/full_metrics['eval_f1']*100:+.2f}%)")
    
    if full_metrics.get('eval_precision') and subsampled_metrics.get('eval_precision'):
        prec_delta = subsampled_metrics['eval_precision'] - full_metrics['eval_precision']
        print(f"  - Precision Delta: {prec_delta:+.4f} ({prec_delta/full_metrics['eval_precision']*100:+.2f}%)")
    
    if full_metrics.get('eval_recall') and subsampled_metrics.get('eval_recall'):
        rec_delta = subsampled_metrics['eval_recall'] - full_metrics['eval_recall']
        print(f"  - Recall Delta: {rec_delta:+.4f} ({rec_delta/full_metrics['eval_recall']*100:+.2f}%)")
    
    print(f"\n{'='*60}\n")

print("Comparison function defined.")

Comparison function defined.


## Model Training & Evaluation

Train BERT models on baseline and augmented datasets, then evaluate all models on the baseline test set


**Use the code in the following cell to train the models on the full datasets. If you want to train on subsampled datasets, instead use the commmented out cell located 2 cells down.**

In [16]:
# print("\n" + "="*70)
# print("STEP 3: TRAINING MODELS ON BASELINE AND AUGMENTED DATASETS")
# print("="*70)

# # Dictionary to store all models and results
# all_models = {}
# all_results = {}
# baseline_model = create_bert_ner_model(num_labels=9)

# # Prepare test dataset (same for all models)
# print("\nPreparing test dataset...")
# test_dataset = baseline_model.prepare_dataset(baseline_dataset['test'])

# # 1. Train on BASELINE dataset
# print("\n" + "-"*70)
# print("Training Model 1: Baseline (No Augmentation)")
# print("-"*70)

# baseline_train_ds = baseline_model.prepare_dataset(baseline_dataset['train'])
# baseline_val_ds = baseline_model.prepare_dataset(baseline_dataset['validation'])

# baseline_output_dir = "./saved_models/bert_baseline_conll2003"
# os.makedirs(baseline_output_dir, exist_ok=True)

# start_time = time.time()
# baseline_train_results = baseline_model.train(
#     train_dataset=baseline_train_ds,
#     eval_dataset=baseline_val_ds,
#     output_dir=baseline_output_dir,
#     num_epochs=3,
#     per_device_batch_size=32,
#     learning_rate=2e-5,
#     weight_decay=0.01,
#     warmup_steps=500,
#     logging_steps=100,
#     eval_strategy="epoch",
#     save_strategy="epoch"
# )
# baseline_training_time = time.time() - start_time
# baseline_model.save_model(baseline_output_dir)
# print(f"✓ Baseline model trained in {baseline_training_time:.2f}s")

# all_models['baseline'] = baseline_model
# all_results['baseline'] = {
#     'training_time': baseline_training_time,
#     'train_samples': len(baseline_dataset['train']),
#     'val_samples': len(baseline_dataset['validation'])
# }

# # 2. Train on AUGMENTED datasets
# for method, aug_dataset in augmented_datasets.items():
#     print("\n" + "-"*70)
#     print(f"Training Model {len(all_models) + 1}: {method.upper()} Augmented Dataset")
#     print("-"*70)
    
#     model = create_bert_ner_model(num_labels=9)
#     train_ds = model.prepare_dataset(aug_dataset['train'])
#     val_ds = model.prepare_dataset(aug_dataset['validation'])
    
#     output_dir = f"./saved_models/bert_{method}_conll2003"
#     os.makedirs(output_dir, exist_ok=True)
    
#     start_time = time.time()
#     train_results = model.train(
#         train_dataset=train_ds,
#         eval_dataset=val_ds,
#         output_dir=output_dir,
#         num_epochs=3,
#         per_device_batch_size=32,
#         learning_rate=2e-5,
#         weight_decay=0.01,
#         warmup_steps=500,
#         logging_steps=100,
#         eval_strategy="epoch",
#         save_strategy="epoch"
#     )
#     training_time = time.time() - start_time
#     model.save_model(output_dir)
#     print(f"✓ {method.upper()} model trained in {training_time:.2f}s")
    
#     all_models[method] = model
#     all_results[method] = {
#         'training_time': training_time,
#         'train_samples': len(aug_dataset['train']),
#         'val_samples': len(aug_dataset['validation']),
#         'corruption_metrics': corruption_metrics[method]
#     }

# print("\n" + "="*70)
# print("✓ All models trained successfully!")
# print("="*70)

In [17]:
print("\n" + "="*70)
print("STEP 3: TRAINING MODELS ON 10% SUBSAMPLED DATA")
print("="*70)

# Dictionary to store all models and results
all_models = {}
all_results = {}
baseline_model = create_bert_ner_model(num_labels=9)

# Prepare test dataset (same for all models - NOT subsampled)
print("\nPreparing test dataset (full size)...")
test_dataset = baseline_model.prepare_dataset(baseline_dataset['test'])

# Function to subsample datasets
def subsample_hf_dataset(dataset, sample_rate=0.1):
    """Subsample a HuggingFace dataset by randomly selecting a fraction of samples."""
    import random
    num_samples = len(dataset)
    num_to_select = max(1, int(num_samples * sample_rate))
    indices = random.sample(range(num_samples), num_to_select)
    return dataset.select(indices)

# 1. Train on BASELINE dataset (10% subsampled)
print("\n" + "-"*70)
print("Training Model 1: Baseline (No Augmentation, 10% Subsampled)")
print("-"*70)

# Subsample training and validation datasets to 10%
baseline_train_subsampled = subsample_hf_dataset(baseline_dataset['train'], sample_rate=0.1)
baseline_val_subsampled = subsample_hf_dataset(baseline_dataset['validation'], sample_rate=0.1)

print(f"Original train size: {len(baseline_dataset['train'])}, Subsampled: {len(baseline_train_subsampled)}")
print(f"Original val size: {len(baseline_dataset['validation'])}, Subsampled: {len(baseline_val_subsampled)}")

baseline_train_ds = baseline_model.prepare_dataset(baseline_train_subsampled)
baseline_val_ds = baseline_model.prepare_dataset(baseline_val_subsampled)

baseline_output_dir = "./saved_models/bert_baseline_conll2003"
os.makedirs(baseline_output_dir, exist_ok=True)

start_time = time.time()
baseline_train_results = baseline_model.train(
    train_dataset=baseline_train_ds,
    eval_dataset=baseline_val_ds,
    output_dir=baseline_output_dir,
    num_epochs=3,
    per_device_batch_size=32,
    learning_rate=2e-5,
    weight_decay=0.01,
    warmup_steps=500,
    logging_steps=100,
    eval_strategy="epoch",
    save_strategy="epoch"
)
baseline_training_time = time.time() - start_time
baseline_model.save_model(baseline_output_dir)
print(f"✓ Baseline model trained in {baseline_training_time:.2f}s")

all_models['baseline'] = baseline_model
all_results['baseline'] = {
    'training_time': baseline_training_time,
    'train_samples': len(baseline_train_subsampled),
    'val_samples': len(baseline_val_subsampled)
}

# 2. Train on AUGMENTED datasets (10% subsampled)
for method, aug_dataset in augmented_datasets.items():
    print("\n" + "-"*70)
    print(f"Training Model {len(all_models) + 1}: {method.upper()} Augmented Dataset (10% Subsampled)")
    print("-"*70)
    
    # Subsample augmented training and validation datasets to 10%
    aug_train_subsampled = subsample_hf_dataset(aug_dataset['train'], sample_rate=0.1)
    aug_val_subsampled = subsample_hf_dataset(aug_dataset['validation'], sample_rate=0.1)
    
    print(f"Original train size: {len(aug_dataset['train'])}, Subsampled: {len(aug_train_subsampled)}")
    print(f"Original val size: {len(aug_dataset['validation'])}, Subsampled: {len(aug_val_subsampled)}")
    
    model = create_bert_ner_model(num_labels=9)
    train_ds = model.prepare_dataset(aug_train_subsampled)
    val_ds = model.prepare_dataset(aug_val_subsampled)
    
    output_dir = f"./saved_models/bert_{method}_conll2003"
    os.makedirs(output_dir, exist_ok=True)
    
    start_time = time.time()
    train_results = model.train(
        train_dataset=train_ds,
        eval_dataset=val_ds,
        output_dir=output_dir,
        num_epochs=3,
        per_device_batch_size=32,
        learning_rate=2e-5,
        weight_decay=0.01,
        warmup_steps=500,
        logging_steps=100,
        eval_strategy="epoch",
        save_strategy="epoch"
    )
    training_time = time.time() - start_time
    model.save_model(output_dir)
    print(f"✓ {method.upper()} model trained in {training_time:.2f}s")
    
    all_models[f"{method}"] = model
    all_results[f"{method}"] = {
        'training_time': training_time,
        'train_samples': len(aug_train_subsampled),
        'val_samples': len(aug_val_subsampled),
        'corruption_metrics': corruption_metrics[method]
    }

print("\n" + "="*70)
print("✓ All models trained successfully on 10% subsampled data!")
print("="*70)



STEP 3: TRAINING MODELS ON 10% SUBSAMPLED DATA


Some weights of BertForTokenClassification were not initialized from the model checkpoint at bert-base-cased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.



Preparing test dataset (full size)...


Tokenizing and aligning labels:   0%|          | 0/3453 [00:00<?, ? examples/s]


----------------------------------------------------------------------
Training Model 1: Baseline (No Augmentation, 10% Subsampled)
----------------------------------------------------------------------
Original train size: 14041, Subsampled: 1404
Original val size: 3250, Subsampled: 325


...

Some weights of BertForTokenClassification were not initialized from the model checkpoint at bert-base-cased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Tokenizing and aligning labels:   0%|          | 0/1404 [00:00<?, ? examples/s]

Tokenizing and aligning labels:   0%|          | 0/325 [00:00<?, ? examples/s]

Epoch,Training Loss,Validation Loss
1,No log,1.944426
2,No log,1.049795
3,1.759500,0.597388


✓ EDA model trained in 237.36s

✓ All models trained successfully on 10% subsampled data!


*Type "3" and Enter if prompted by the cell below*

In [18]:
print("\n" + "="*70)
print("STEP 4: EVALUATING ALL MODELS ON BASELINE TEST SET")
print("="*70)

# Evaluate all models on the SAME test set (baseline)
LABEL_MAPPING = {
    'LABEL_0': "O",
    'LABEL_1': "B-PER",
    'LABEL_2': "I-PER",
    'LABEL_3': "B-ORG",
    'LABEL_4': "I-ORG",
    'LABEL_5': "B-LOC",
    'LABEL_6': "I-LOC",
    'LABEL_7': "B-MISC",
    'LABEL_8': "I-MISC"
}

for model_name, model in all_models.items():
    print(f"\nEvaluating {model_name.upper()} model...")
    
    # Move to CPU for evaluation
    model.model.to("cpu")
    
    # Evaluate
    start_time = time.time()
    eval_results = model.evaluate(test_dataset)
    eval_time = time.time() - start_time
    
    # Get predictions for per-type metrics
    true_labels, predictions = model.get_predictions(test_dataset, label_mapping=LABEL_MAPPING)
    per_type_scores = per_type_f1_scores(true_labels, predictions)
    
    # Store results
    all_results[model_name]['eval_time'] = eval_time
    all_results[model_name]['test_samples'] = len(baseline_dataset['test'])
    all_results[model_name]['eval_loss'] = eval_results.get('eval_loss', None)
    all_results[model_name]['eval_f1'] = eval_results.get('eval_f1', None)
    all_results[model_name]['eval_precision'] = eval_results.get('eval_precision', None)
    all_results[model_name]['eval_recall'] = eval_results.get('eval_recall', None)
    all_results[model_name]['per_type_f1_scores'] = per_type_scores
    
    print(f"  ✓ Evaluation complete in {eval_time:.2f}s")
    print(f"    Loss: {all_results[model_name]['eval_loss']:.4f}")
    print(f"    F1: {all_results[model_name]['eval_f1']:.4f}")
    print(f"    Precision: {all_results[model_name]['eval_precision']:.4f}")
    print(f"    Recall: {all_results[model_name]['eval_recall']:.4f}")

print("\n" + "="*70)
print("✓ All models evaluated!")
print("="*70)



STEP 4: EVALUATING ALL MODELS ON BASELINE TEST SET

Evaluating BASELINE model...


/content/ner_data_augmentation_error_analysis/src/model/bert_model.py:556: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results
wandb: Enter your choice:wandb: You chose "Don't visualize my results"
wandb: Using W&B in offline mode.
wandb: W&B API key is configured. Use `wandb login --relogin` to force relogin


  ✓ Evaluation complete in 61.34s
    Loss: 0.6597
    F1: 0.7223
    Precision: 0.7149
    Recall: 0.7886

Evaluating ENTITY_REPLACEMENT model...


/content/ner_data_augmentation_error_analysis/src/model/bert_model.py:556: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


...

  ✓ Evaluation complete in 31.32s
    Loss: 0.7575
    F1: 0.6592
    Precision: 0.6588
    Recall: 0.7587

Evaluating CONTEXTUAL_MLM model...


/content/ner_data_augmentation_error_analysis/src/model/bert_model.py:556: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


  ✓ Evaluation complete in 30.71s
    Loss: 0.6574
    F1: 0.7344
    Precision: 0.7434
    Recall: 0.7915

Evaluating EDA model...


/content/ner_data_augmentation_error_analysis/src/model/bert_model.py:556: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


  ✓ Evaluation complete in 30.55s
    Loss: 0.6460
    F1: 0.7428
    Precision: 0.7468
    Recall: 0.7970

✓ All models evaluated!


In [ ]:
print("\n" + "="*70)
print("STEP 5: COMPREHENSIVE RESULTS SUMMARY")
print("="*70)

# 1. Training Summary
print("\n" + "-"*70)
print("TRAINING SUMMARY")
print("-"*70)
print(f"{'Model':<20} {'Dataset Size':<15} {'Training Time (s)':<20}")
print(f"{'-'*70}")
for model_name, results in all_results.items():
    dataset_size = results['train_samples']
    training_time = results['training_time']
    print(f"{model_name:<20} {dataset_size:<15} {training_time:<20.2f}")

# 2. Augmentation Quality Summary
print("\n" + "-"*70)
print("AUGMENTATION QUALITY METRICS")
print("-"*70)
print(f"{'Method':<20} {'Span Corruption':<20} {'Label Flip Rate':<20}")
print(f"{'-'*70}")
for method, metrics in corruption_metrics.items():
    span_corr = metrics['span_corruption']['span_corruption_rate']
    label_flip = metrics['label_flip']['label_flip_rate']
    print(f"{method:<20} {span_corr:<20.4f} {label_flip:<20.4f}")

# 3. Evaluation Results on Baseline Test Set
print("\n" + "-"*70)
print("EVALUATION RESULTS (All Models Tested on Baseline Test Set)")
print("-"*70)
print(f"{'Model':<20} {'F1':<12} {'Precision':<12} {'Recall':<12} {'Loss':<12}")
print(f"{'-'*70}")
for model_name, results in all_results.items():
    f1 = results.get('eval_f1', 0.0)
    precision = results.get('eval_precision', 0.0)
    recall = results.get('eval_recall', 0.0)
    loss = results.get('eval_loss', 0.0)
    print(f"{model_name:<20} {f1:<12.4f} {precision:<12.4f} {recall:<12.4f} {loss:<12.4f}")

# 4. Per-Entity-Type Results Comparison
print("\n" + "-"*70)
print("PER-ENTITY-TYPE F1 SCORES (All Models on Baseline Test Set)")
print("-"*70)

entity_types = set()
for model_name, results in all_results.items():
    per_type = results.get('per_type_f1_scores', {})
    entity_types.update(per_type.keys())

entity_types = sorted([e for e in entity_types if e != 'overall'])

print(f"\n{'Entity Type':<15}", end="")
for model_name in all_results.keys():
    print(f"{model_name:<18}", end="")
print()
print("-" * (15 + 18 * len(all_results)))

for entity_type in entity_types:
    print(f"{entity_type:<15}", end="")
    for model_name, results in all_results.items():
        per_type = results.get('per_type_f1_scores', {})
        f1 = per_type.get(entity_type, {}).get('f1', 0.0)
        print(f"{f1:<18.4f}", end="")
    print()

print("\nOVERALL")
print(f"{'F1 Scores':<15}", end="")
for model_name, results in all_results.items():
    per_type = results.get('per_type_f1_scores', {})
    f1 = per_type.get('overall', {}).get('f1', 0.0)
    print(f"{f1:<18.4f}", end="")
print()

# 5. Performance Delta Analysis
print("\n" + "-"*70)
print("PERFORMANCE DELTA vs BASELINE")
print("-"*70)
baseline_f1 = all_results['baseline'].get('eval_f1', 0.0)

for model_name in all_results.keys():
    if model_name != 'baseline':
        model_f1 = all_results[model_name].get('eval_f1', 0.0)
        f1_delta = model_f1 - baseline_f1
        f1_pct_change = (f1_delta / baseline_f1 * 100) if baseline_f1 > 0 else 0
        
        model_recall = all_results[model_name].get('eval_recall', 0.0)
        baseline_recall = all_results['baseline'].get('eval_recall', 0.0)
        recall_delta = model_recall - baseline_recall
        
        print(f"\n{model_name.upper()}:")
        print(f"  F1 Delta: {f1_delta:+.4f} ({f1_pct_change:+.2f}%)")
        print(f"  Recall Delta: {recall_delta:+.4f}")

# 6. Save comprehensive results to JSON
print("\n" + "="*70)
print("SAVING RESULTS")
print("="*70)

results_dir = "./results"
os.makedirs(results_dir, exist_ok=True)

# Prepare results for JSON serialization
json_results = {
    'experiment_type': 'Augmentation Comparison Study',
    'baseline_dataset_sizes': {
        'train': len(baseline_dataset['train']),
        'validation': len(baseline_dataset['validation']),
        'test': len(baseline_dataset['test'])
    },
    'models': {}
}

for model_name, results in all_results.items():
    json_results['models'][model_name] = {
        'training': {
            'training_time': float(results.get('training_time', 0)),
            'train_samples': results.get('train_samples', 0),
            'val_samples': results.get('val_samples', 0),
        },
        'augmentation_quality': results.get('corruption_metrics', {}),
        'evaluation': {
            'eval_time': float(results.get('eval_time', 0)),
            'test_samples': results.get('test_samples', 0),
            'eval_loss': float(results.get('eval_loss', 0)) if results.get('eval_loss') else None,
            'eval_f1': float(results.get('eval_f1', 0)) if results.get('eval_f1') else None,
            'eval_precision': float(results.get('eval_precision', 0)) if results.get('eval_precision') else None,
            'eval_recall': float(results.get('eval_recall', 0)) if results.get('eval_recall') else None,
        },
        'per_type_f1_scores': results.get('per_type_f1_scores', {})
    }

results_path = os.path.join(results_dir, "augmentation_experiment.json")
with open(results_path, "w") as f:
    json.dump(json_results, f, indent=2)

print(f"\n✓ Results saved to {results_path}")
print("\n✅ Augmentation experiment completed successfully!")



STEP 5: COMPREHENSIVE RESULTS SUMMARY

----------------------------------------------------------------------
TRAINING SUMMARY
----------------------------------------------------------------------
Model                Dataset Size    Training Time (s)   
----------------------------------------------------------------------
baseline             1404            136.29              
entity_replacement   1404            141.72              
back_translation     1404            248.61              
contextual_mlm       1404            256.94              
eda                  1404            237.36              

----------------------------------------------------------------------
AUGMENTATION QUALITY METRICS
----------------------------------------------------------------------
Method               Span Corruption      Label Flip Rate     
----------------------------------------------------------------------
entity_replacement   0.4723               0.0295              
back_translat

Optional Cell for Downloading any generated files

In [20]:
# import shutil
# from google.colab import files

# # Zip the folder (creates augmented_datasets.zip in ./data)
# shutil.make_archive("./saved_models", "zip", "./saved_models")

# # Download it
# files.download("./saved_models.zip")